# MCP Gateway & Registry Workshop

Deploy the [MCP Gateway & Registry](https://github.com/agentic-community/mcp-gateway-registry) on your EKS cluster. This platform centralizes access to MCP servers and AI agents with authentication, governance, and dynamic tool discovery.

| Module | Description | Dependency |
|--------|-------------|------------|
| **Module 0** | Clone the upstream chart and build dependencies | |
| **Module 1** | Deploy the full stack (MongoDB, Keycloak, Registry, Auth Server) | Module 0 |
| **Module 2** | Access the Registry UI and retrieve credentials | Module 1 |

## Prerequisites

* This notebook **must be run** on the ML Ops desktop created via [quick start setup](../../../README.md#quick-start-basic). This notebook can also be used on an ML Ops desktop created via [advanced setup](../../../README.md#advanced-setup), but the user will need to appropriately adapt the notebook for use with the advanced setup.
* `helm` CLI v3.0+ installed.
* `kubectl` configured to access your EKS cluster.

## Architecture

The stack deploys the following components into the `mcp-gateway` namespace:

```
┌──────────────────────────────────────────────────────────────────────┐
│                           EKS Cluster                               │
│                                                                      │
│  ┌────────────────────────────────────────────────────────────────┐  │
│  │                    mcp-gateway namespace                       │  │
│  │                                                                │  │
│  │   ┌───────────────┐                                            │  │
│  │   │ Nginx Gateway │─── /registry/*  ──► Registry (8000)       │  │
│  │   │   (8080)      │─── /auth-server/*► Auth Server (8888)     │  │
│  │   │               │─── /keycloak/*  ──► Keycloak (80)         │  │
│  │   │               │─── /mcpgw/*     ──► MCPGW (8003)         │  │
│  │   └───────────────┘                                            │  │
│  │                                                                │  │
│  │   ┌─────────────┐    ┌──────────────┐    ┌────────────────┐   │  │
│  │   │  Keycloak   │    │  Auth Server  │    │   Registry     │   │  │
│  │   │   (IdP)     │◄──►│  (OAuth)      │◄──►│   (Web UI)     │   │  │
│  │   └──────┬──────┘    └──────────────┘    └────────────────┘   │  │
│  │          │                                                     │  │
│  │   ┌──────▼──────┐    ┌──────────────┐                         │  │
│  │   │ PostgreSQL  │    │   MongoDB     │                         │  │
│  │   │ (Keycloak)  │    │  (Registry)   │                         │  │
│  │   └─────────────┘    └──────────────┘                         │  │
│  │                                                                │  │
│  │   ┌──────────────┐                                             │  │
│  │   │   MCPGW      │                                             │  │
│  │   │ (MCP Server) │                                             │  │
│  │   └──────────────┘                                             │  │
│  └────────────────────────────────────────────────────────────────┘  │
└──────────────────────────────────────────────────────────────────────┘
         │
         ▼
   port-forward
   :8080 (Gateway)
```

---
## Module 0: Clone Chart and Build Dependencies

The MCP Gateway Registry Helm chart uses local file references for its sub-charts, so we need to clone the upstream repository and build dependencies from source.

### Step 0.1: Set Up Paths

In [ ]:
import os

REPO_DIR = os.path.expanduser("~/amazon-eks-machine-learning-with-terraform-and-kubeflow")
EXAMPLE_DIR = os.path.join(REPO_DIR, "examples/agentic/mcp-gateway-registry")
UPSTREAM_DIR = os.path.expanduser("~/mcp-gateway-registry")
CHART_DIR = os.path.join(UPSTREAM_DIR, "charts/mcp-gateway-registry-stack")

os.environ["REPO_DIR"] = REPO_DIR
os.environ["EXAMPLE_DIR"] = EXAMPLE_DIR
os.environ["UPSTREAM_DIR"] = UPSTREAM_DIR
os.environ["CHART_DIR"] = CHART_DIR

NAMESPACE = "mcp-gateway"
os.environ["NAMESPACE"] = NAMESPACE

print(f"Example Dir:  {EXAMPLE_DIR}")
print(f"Upstream Dir: {UPSTREAM_DIR}")
print(f"Chart Dir:    {CHART_DIR}")
print(f"Namespace:    {NAMESPACE}")

### Step 0.2: Clone the Upstream Repository

In [ ]:
%%bash
RELEASE_TAG="v1.0.16"

if [  ! -d "$UPSTREAM_DIR" ]; then
  # Use --depth 1 if you only need the files at this tag (faster)
  git clone --branch "$RELEASE_TAG" --depth 1 https://github.com/agentic-community/mcp-gateway-registry.git "$UPSTREAM_DIR"
else
  cd "$UPSTREAM_DIR"
  # --tags ensures the specific tag is downloaded even if it's new
  git fetch origin --tags
  git checkout "$RELEASE_TAG"
fi


### Step 0.3: Add Nginx Gateway Template

Copy the nginx gateway template into the cloned chart. This adds an in-cluster nginx reverse proxy that provides a single entry point for all services via path-based routing, so only one `kubectl port-forward` is needed.

In [ ]:
%%bash
cp $EXAMPLE_DIR/templates/nginx-gateway.yaml $CHART_DIR/templates/
echo "Copied nginx-gateway.yaml to chart templates"

### Step 0.4: Build Helm Dependencies

In [ ]:
%%bash
cd $CHART_DIR
helm dependency build
helm dependency update
echo "Dependencies built successfully"

---
## Module 1: Deploy the Full Stack

Install the MCP Gateway Registry stack using the workshop values file. This deploys MongoDB, Keycloak, the Registry, Auth Server, and MCPGW into the `mcp-gateway` namespace.

All ingress is disabled — we use `kubectl port-forward` to access services.

### Step 1.1: Generate a MongoDB Password

In [ ]:
import secrets
import string

MONGO_USERNAME = 'mcpgateway'
MONGO_PASSWORD = ''.join(secrets.choice(string.ascii_letters + string.digits) for _ in range(32))
os.environ["MONGO_USERNAME"] = MONGO_USERNAME
os.environ["MONGO_PASSWORD"] = MONGO_PASSWORD
print("MongoDB credentials generated (stored in environment)")

### Step 1.2: Install the Helm Chart

This takes **5-10 minutes**. Keycloak and MongoDB need to initialize before dependent services start.

In [ ]:
%%bash
cd $CHART_DIR
helm install mcp-gateway-registry \
  -n $NAMESPACE --create-namespace \
  -f $EXAMPLE_DIR/values.yaml \
  --set mongodb.password="$MONGO_PASSWORD" \
  --set mongodb-configure.mongodb.password="$MONGO_PASSWORD" \
  --set mongodb-configure.mongodb.username="$MONGO_USERNAME" \
  --set mongodb.user="$MONGO_USERNAME" \
  .

### Step 1.3: Patch Registry Admin Password

The upstream chart has a bug where the `keycloak-configure` job generates the admin password and saves it to `registry-login-credentials`, but never wires it into the registry deployment. We wait for the job to complete, then patch `registry-secret` with the `ADMIN_PASSWORD` the container expects.

In [ ]:
%%bash
echo "Waiting for keycloak-configure job to complete..."
kubectl wait --for=condition=complete job/setup-keycloak -n $NAMESPACE --timeout=600s

echo "Patching registry-secret with ADMIN_PASSWORD..."
ADMIN_PASS=$(kubectl get secret registry-login-credentials -n $NAMESPACE -o jsonpath='{.data.REGISTRY_ADMIN_PASSWORD}' | base64 -d)
kubectl patch secret registry-secret -n $NAMESPACE -p '{"data":{"ADMIN_PASSWORD":"'$(echo -n "$ADMIN_PASS" | base64)'"}}'

echo "Restarting registry deployment..."
kubectl rollout restart deployment registry -n $NAMESPACE
echo "Done."

### Step 1.4: Wait for Pods to Be Ready

Monitor the rollout. All pods should now come up successfully.

In [ ]:
%%bash
echo "Waiting for deployments in $NAMESPACE namespace..."
for deploy in $(kubectl get deployments -n $NAMESPACE -o jsonpath='{.items[*].metadata.name}'); do
  echo "Waiting for deployment/$deploy..."
  kubectl rollout status deployment/$deploy -n $NAMESPACE --timeout=600s
done
echo ""
echo "Waiting for statefulsets in $NAMESPACE namespace..."
for sts in $(kubectl get statefulsets -n $NAMESPACE -o jsonpath='{.items[*].metadata.name}'); do
  echo "Waiting for statefulset/$sts..."
  kubectl rollout status statefulset/$sts -n $NAMESPACE --timeout=600s
done
echo ""
kubectl get pods -n $NAMESPACE

---
## Module 2: Access the Registry UI

With all services running, use port-forwarding to access the Registry web UI.

### Step 2.1: Retrieve Keycloak Credentials

The `keycloak-configure` job creates a user for the Registry. Retrieve the credentials from the job logs.

In [ ]:
%%bash
echo "Looking for completed keycloak-configure job..."
COMPLETED_POD=$(kubectl get pods -n $NAMESPACE -l job-name=setup-keycloak \
  --field-selector=status.phase==Succeeded -o jsonpath='{.items[0].metadata.name}' 2>/dev/null)

if [ -z "$COMPLETED_POD" ]; then
  echo "Keycloak configure job has not completed yet. Re-run this cell after pods are ready."
else
  echo "Credentials from keycloak-configure job:"
  echo "=========================================="
  kubectl logs -n $NAMESPACE $COMPLETED_POD --tail=20
fi

### Step 2.2: Port-Forward the Gateway

An nginx gateway pod runs inside the cluster and routes all traffic to the backend services by path. You only need a single port-forward.

Run the following command in a **separate terminal** on the ML Ops desktop, then open the UI in your browser.

In [ ]:
%%bash
echo "Run the following command in a separate terminal on the ML Ops desktop:"
echo ""
echo "  kubectl port-forward svc/nginx-gateway -n $NAMESPACE 8080:8080"
echo ""
echo "Then open: http://localhost:8080/registry/"
echo ""
echo "Log in with the credentials from Step 2.1."

### Step 2.3: Explore the Registry

Once logged in, you can:

- **Register MCP Servers** — Add your MCP servers (e.g., EKS MCP Server, custom tools) to the registry for centralized access and governance.
- **Register A2A Agents** — Register AI agents that support the Agent-to-Agent protocol for discovery and communication.
- **Configure Access Control** — Set up groups and fine-grained permissions for users and service accounts via the IAM Settings page.
- **Search & Discover** — Use semantic search to find registered servers, tools, and agents by natural language description.

For full documentation on configuring the registry, see the [MCP Gateway & Registry docs](https://github.com/agentic-community/mcp-gateway-registry).

---
## Troubleshooting

| Issue | Cause | Solution |
|-------|-------|----------|
| Keycloak pod stuck in `CrashLoopBackOff` | PostgreSQL not ready | Wait for the PostgreSQL pod to be `Running` first |
| Auth server not starting | `keycloak-configure` job not completed | Check job status: `kubectl get jobs -n mcp-gateway` |
| Cannot log in to Registry UI | Port-forward not running | Ensure `kubectl port-forward svc/nginx-gateway -n mcp-gateway 8080:8080` is active |
| MongoDB pod pending | No storage class available | Verify a default StorageClass exists: `kubectl get sc` |

In [ ]:
# Useful troubleshooting commands
!kubectl get pods -n mcp-gateway
!kubectl get jobs -n mcp-gateway
!kubectl logs -n mcp-gateway -l app.kubernetes.io/name=registry --tail=30

---
## Cleanup

Uninstall the stack and delete the namespace.

In [ ]:
%%bash
helm uninstall mcp-gateway-registry -n $NAMESPACE
kubectl delete namespace $NAMESPACE
echo "Cleanup complete"